In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import os
CODE = "/content/drive/MyDrive/esci/code"
print(sorted(os.listdir(CODE)))                                   # should show cross_encoder.py (no "(1)")
print("has query_to_text:", "def query_to_text" in open(f"{CODE}/cross_encoder.py").read())
print("has key_value:",     "key_value" in open(f"{CODE}/cross_encoder.py").read())

Mounted at /content/drive
['__pycache__', 'cross_encoder.py', 'metrics.py', 'train_cross_encoder.py']
has query_to_text: True
has key_value: True


In [1]:
from google.colab import drive
drive.mount('/content/drive')
import sys, importlib, numpy as np, pandas as pd
CODE = "/content/drive/MyDrive/esci/code"; sys.path.insert(0, CODE)
import cross_encoder as C; importlib.reload(C)

BASE = "/content/drive/MyDrive/esci"
q = pd.read_parquet(f"{BASE}/dataset_v2.3/esci_electronics_v2.3_queries.parquet")

# M3 category stand-in: ONLY for Rule A queries (category found in the query text).
# Rule B categories come from labelled products -> would leak the answer, so no category there.
CAT = {"mobile_computing": "Mobile & Computing", "components_accessories": "Components & Accessories",
       "home_electronics": "Home Electronics", "imaging": "Cameras & Imaging", "audio": "Audio",
       "computer_peripherals": "Computer Peripherals", "gaming": "Gaming", "smart_home": "Smart Home"}
q["category"] = np.where(q["rule"] == "A", q["scope_group"].map(CAT), None)

q["focused"]     = [C.query_to_text(x) for x in q["query"]]
q["focused_cat"] = [C.query_to_text(x, category=c) for x, c in zip(q["query"], q["category"])]
q["changed"]     = q["focused"] != q["query"].map(C._clean)

RULES = {"price_limit": C._PRICE_LIMIT, "price_alone": C._PRICE_ALONE, "price_word": C._PRICE_WORDS,
         "platform": C._PLATFORM, "delivery": C._DELIVERY}
q["removed"] = [",".join(n for n, rx in RULES.items() if rx.search(x)) for x in q["query"]]

print("queries changed by query_to_text:", int(q["changed"].sum()), "of", len(q),
      f"({q['changed'].mean():.1%})")
print("\nby split:\n", q.groupby("split")["changed"].agg(["sum", "mean"]).round(3))
print("\nwhich rule fired:\n", q.loc[q["changed"], "removed"].value_counts())
print("\nqueries with a category stand-in (Rule A):", int(q["category"].notna().sum()))
print("\n--- 30 changed examples (please read them) ---")
print(q.loc[q["changed"], ["query", "focused"]].sample(min(30, int(q["changed"].sum())), random_state=1).to_string())

q[["query_id", "query", "split", "rule", "category", "focused", "focused_cat", "changed", "removed"]] \
  .to_csv(f"{BASE}/results/query_text_check_v1.csv", index=False)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
queries changed by query_to_text: 53 of 5665 (0.9%)

by split:
        sum   mean
split            
test    12  0.009
train   41  0.009

which rule fired:
 removed
price_word                            34
price_limit,price_alone                8
price_limit                            5
price_alone                            3
price_alone,price_word                 1
price_limit,price_alone,price_word     1
delivery                               1
Name: count, dtype: int64

queries with a category stand-in (Rule A): 4189

--- 30 changed examples (please read them) ---
                                                                                                                                                     query                                                                                                                                   focused
2660